# Lab CosyVoice3 + Stage 1–2 manifest

**Model:** Fun-CosyVoice3-0.5B-2512 (docs: https://github.com/FunAudioLLM/CosyVoice)

**Ref:** đã commit trong repo tại `voice_bank/cosy_refs/` (sinh từ Kokoro, mono **16 kHz**). Clone repo là có sẵn — **không** cần upload tay.

## Thứ tự chạy

| Bước | Cell | Việc |
|------|------|------|
| 1 | Cell 1 | Config |
| 2 | Cell 2 | Clone/pull IELTS repo + prepare manifest |
| 3 | Cell 3 | **Verify ref** trong repo (wav + json) |
| 4 | Cell 4 | Clone CosyVoice + pip |
| 5 | Cell 5 | Download weight 2512 |
| 6 | Cell 6 | Smoke zero_shot 1 câu EN |
| 7 | Cell 7 | Render Part 1 từ manifest (limit 4) |
| 8 | Cell 8 | Tracking |
| 9 | Cell 9 | (Tuỳ) instruct nhanh/căng |

Không cài nặng về laptop. Không sửa `display_text`.


## Cell 1 — Config


In [ ]:
# CELL 1
IELTS_REPO = "https://github.com/phamtu2x5/ielts-script2audio.git"
IELTS_DIR = "/content/ielts-script2audio"
COSY_DIR = "/content/CosyVoice"
BRANCH = "main"
print("OK Cell 1")


## Cell 2 — Clone IELTS + prepare manifest


In [ ]:
# CELL 2
import os
from pathlib import Path

if not Path(IELTS_DIR).exists():
    !git clone --branch {BRANCH} {IELTS_REPO} {IELTS_DIR}
else:
    %cd {IELTS_DIR}
    !git pull origin {BRANCH}
%cd {IELTS_DIR}
!pip -q install -e ".[dev]"

Path("outputs").mkdir(exist_ok=True)
!ielts-s2a prepare examples/part1_valid.json --pretty -o outputs/part1_manifest.json

import json
m = json.loads(Path("outputs/part1_manifest.json").read_text())
assert m["validation"]["valid"] is True
print("OK Cell 2 n_requests=", len(m["requests"]))


## Cell 3 — Verify ref đã có trong repo (sau clone)


In [ ]:
# CELL 3 — load committed CosyVoice refs (no manual download)
from pathlib import Path
import json
import wave

IELTS = Path(IELTS_DIR)
ref_dir = IELTS / "voice_bank" / "cosy_refs"
man_path = ref_dir / "refs_manifest.json"
map_path = ref_dir / "voice_map_runtime.json"

assert man_path.is_file(), f"Missing {man_path} — git pull main with committed refs"
assert map_path.is_file(), f"Missing {map_path}"

man = json.loads(man_path.read_text())
vmap = json.loads(map_path.read_text())

# Resolve repo-relative paths -> absolute for CosyVoice / torchaudio
for vp, entry in (vmap.get("mapping") or {}).items():
    rel = entry.get("ref_wav_relative") or entry.get("ref_wav")
    abs_path = (IELTS / rel).resolve() if not Path(rel).is_absolute() else Path(rel)
    # if relative stored as voice_bank/...
    if not abs_path.is_file():
        abs_path = (IELTS / "voice_bank" / "cosy_refs" / Path(rel).name).resolve()
    entry["ref_wav"] = str(abs_path)
    entry["ref_wav_relative"] = str(Path("voice_bank/cosy_refs") / abs_path.name)
    with wave.open(str(abs_path), "rb") as w:
        ch, sw, sr, n = w.getnchannels(), w.getsampwidth(), w.getframerate(), w.getnframes()
    sec = round(n / sr, 2)
    print(vp, abs_path.name, f"ch={ch} sw={sw} sr={sr} sec={sec}")
    assert ch == 1 and sw == 2 and sr == 16000, "ref must be mono int16 16kHz"
    assert sec >= 5.0, f"ref too short: {sec}s"
    assert abs_path.is_file()

# CosyVoice model dir lives under CosyVoice clone (set after Cell 5 download)
vmap["model_dir"] = str(Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B")
runtime_map = ref_dir / "voice_map_runtime.colab.json"
runtime_map.write_text(json.dumps(vmap, ensure_ascii=False, indent=2) + "\n")
print("OK Cell 3 refs ready ->", runtime_map)


## Cell 4 — Clone CosyVoice + requirements (docs official)


In [ ]:
# CELL 4
import os
from pathlib import Path

if not Path(COSY_DIR).exists():
    !git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git {COSY_DIR}
%cd {COSY_DIR}
!git submodule update --init --recursive
!pip -q install -r requirements.txt
!apt-get -qq update && apt-get -qq install -y sox libsox-dev || true
print("OK Cell 4", os.getcwd())


## Cell 5 — Download Fun-CosyVoice3-0.5B-2512


In [ ]:
# CELL 5 — weights (slow first time)
from huggingface_hub import snapshot_download
from pathlib import Path

model_dir = Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B"
model_dir.parent.mkdir(parents=True, exist_ok=True)
snapshot_download(
    "FunAudioLLM/Fun-CosyVoice3-0.5B-2512",
    local_dir=str(model_dir),
)
print("OK Cell 5", model_dir, "exists=", model_dir.exists())


## Cell 6 — Smoke zero_shot EN (docs CosyVoice3)


In [ ]:
# CELL 6 — smoke
import sys
import json
from pathlib import Path
import torchaudio

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

from cosyvoice.cli.cosyvoice import AutoModel

IELTS = Path(IELTS_DIR)
vmap = json.loads((IELTS / "voice_bank/cosy_refs/voice_map_runtime.colab.json").read_text())
model_dir = str(Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B")
entry = vmap["mapping"]["vp_spk_a"]
ref = entry["ref_wav"]
prompt_text = entry["prompt_text"]
assert Path(ref).is_file(), ref

cosyvoice = AutoModel(model_dir=model_dir)
tts_text = "Good morning. This is a short CosyVoice lab test for English."

out_dir = IELTS / "lab_audio/cosyvoice3_smoke"
out_dir.mkdir(parents=True, exist_ok=True)
for i, j in enumerate(cosyvoice.inference_zero_shot(tts_text, prompt_text, ref, stream=False)):
    p = out_dir / f"smoke_zero_shot_{i}.wav"
    torchaudio.save(str(p), j["tts_speech"], cosyvoice.sample_rate)
    print("wrote", p)
print("OK Cell 6 sr=", cosyvoice.sample_rate)
globals()["cosyvoice"] = cosyvoice


## Cell 7 — Render Part 1 manifest (limit 4)


In [ ]:
# CELL 7 — render from Stage-2 manifest
import json
import sys
from pathlib import Path
import torch
import torchaudio

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

from cosyvoice.cli.cosyvoice import AutoModel

IELTS = Path(IELTS_DIR)
manifest = json.loads((IELTS / "outputs/part1_manifest.json").read_text())
vmap = json.loads((IELTS / "voice_bank/cosy_refs/voice_map_runtime.colab.json").read_text())
model_dir = str(Path(COSY_DIR) / "pretrained_models" / "Fun-CosyVoice3-0.5B")

if "cosyvoice" not in globals():
    cosyvoice = AutoModel(model_dir=model_dir)
else:
    cosyvoice = globals()["cosyvoice"]

out_dir = IELTS / "lab_audio/cosyvoice3_part1"
out_dir.mkdir(parents=True, exist_ok=True)

LIMIT = 4
requests = list(manifest.get("requests") or [])
if LIMIT is not None:
    requests = requests[:LIMIT]

report = {
    "backend": "fun-cosyvoice3-0.5b-2512",
    "mode": "zero_shot",
    "ref_source": "kokoro_synthetic_committed",
    "ok_count": 0,
    "fail_count": 0,
    "segments": [],
}

for req in requests:
    seg_id = req["segment_id"]
    vp = req["voice_profile_id"]
    spoken = req.get("spoken_text") or req.get("display_text") or ""
    display = req.get("display_text") or ""
    entry = (vmap.get("mapping") or {}).get(vp) or {}
    ref_wav = entry.get("ref_wav")
    prompt_text = entry.get("prompt_text") or "You are a helpful assistant.<|endofprompt|>Hello."
    item = {
        "segment_id": seg_id,
        "speaker_id": req.get("speaker_id"),
        "voice_profile_id": vp,
        "display_text": display,
        "spoken_text": spoken,
        "ref_wav": ref_wav,
        "status": "PENDING",
        "output_filename": None,
        "error": None,
    }
    try:
        if not ref_wav or not Path(ref_wav).is_file():
            raise FileNotFoundError(f"missing ref for {vp}: {ref_wav}")
        chunks = []
        for j in cosyvoice.inference_zero_shot(spoken, prompt_text, ref_wav, stream=False):
            chunks.append(j["tts_speech"])
        if not chunks:
            raise RuntimeError("no audio")
        audio = torch.cat(chunks, dim=1) if len(chunks) > 1 else chunks[0]
        out_name = f"{seg_id}__{vp}.wav"
        torchaudio.save(str(out_dir / out_name), audio, cosyvoice.sample_rate)
        item["status"] = "EXECUTED"
        item["output_filename"] = out_name
        report["ok_count"] += 1
    except Exception as e:
        item["status"] = "FAILED"
        item["error"] = f"{type(e).__name__}: {e}"
        report["fail_count"] += 1
    report["segments"].append(item)
    print(item["segment_id"], item["status"], item.get("error") or item.get("output_filename"))

(out_dir / "lab_render_report.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2) + "\n"
)
print("OK Cell 7 ok=", report["ok_count"], "fail=", report["fail_count"])


## Cell 8 — Tracking


In [ ]:
# CELL 8 — track
from pathlib import Path
from IPython.display import Audio, display
import json

rep = Path(IELTS_DIR) / "lab_audio/cosyvoice3_part1/lab_render_report.json"
assert rep.is_file(), "Chay Cell 7 truoc"
report = json.loads(rep.read_text())
audio_dir = rep.parent
for i, seg in enumerate(report.get("segments") or [], 1):
    print("=" * 72)
    print(f"[{i}] {seg.get('segment_id')} | {seg.get('speaker_id')} | {seg.get('status')}")
    print("DISPLAY:", seg.get("display_text"))
    print("SPOKEN :", seg.get("spoken_text"))
    print("REF    :", seg.get("ref_wav"))
    fn = seg.get("output_filename")
    wav = audio_dir / fn if fn else None
    if wav and wav.is_file():
        display(Audio(filename=str(wav)))
    elif seg.get("error"):
        print("ERROR", seg["error"])
print("OK Cell 8")


## Cell 9 — (Tuỳ) instruct nhanh/căng


In [ ]:
# CELL 9 — instruct smoke
import sys
import json
from pathlib import Path
import torchaudio

sys.path.insert(0, str(Path(COSY_DIR) / "third_party" / "Matcha-TTS"))
sys.path.insert(0, COSY_DIR)

from cosyvoice.cli.cosyvoice import AutoModel

IELTS = Path(IELTS_DIR)
vmap = json.loads((IELTS / "voice_bank/cosy_refs/voice_map_runtime.colab.json").read_text())
if "cosyvoice" not in globals():
    cosyvoice = AutoModel(model_dir=str(Path(COSY_DIR) / "pretrained_models/Fun-CosyVoice3-0.5B"))

ref = vmap["mapping"]["vp_spk_b"]["ref_wav"]
tts_text = "I need this sorted out today. The booking is wrong and I am not happy."
instruct = vmap.get("tense_instruct") or (
    "You are a helpful assistant. Speak slightly faster, more urgent and tense.<|endofprompt|>"
)
out_dir = IELTS / "lab_audio/cosyvoice3_smoke"
out_dir.mkdir(parents=True, exist_ok=True)
for i, j in enumerate(cosyvoice.inference_instruct2(tts_text, instruct, ref, stream=False)):
    p = out_dir / f"smoke_instruct_tense_{i}.wav"
    torchaudio.save(str(p), j["tts_speech"], cosyvoice.sample_rate)
    print("wrote", p)
print("OK Cell 9")


Xong lab. not_selected. Ref Kokoro synthetic committed for Colab clone.
